# v0.2.8 live smoke test (one terrestrial zone + one aquatic zone)

Runs the **v0.2.8 engine** on `EMU_Deccan_forest` (terrestrial) and `Lake_Suman` (aquatic) and compares the Earth Engine
outputs with the numpy definitions. It does **not** run the 15 tiles and does **not** touch the headline aggregation or `main`.

Run the cells in order. Send back: the two `*_audit.csv/json/md` files and `parity.md/json` (the last cell zips them).
Quota: this is heavier than a normal single-tile run for the `natural_habitat` / `ndvi` / `chm` / `bii` / `ghm` reference cells and the
water-body vectorisation. Check your Earth Engine quota first; keep the runtime alive between cells.

In [ ]:
# 1. Clone / update the v0.2.8 BRANCH (main is untouched). Safe to re-run.
import os, glob
REPO_DIR = '/content/reference-benchmarking'
BRANCH = 'v0.2.8-contract'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH https://github.com/G-auravSingh/reference-benchmarking.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $BRANCH
    !git -C $REPO_DIR checkout $BRANCH
    !git -C $REPO_DIR pull --ff-only origin $BRANCH
PKG_DIR = sorted(glob.glob(f'{REPO_DIR}/darukaa_reference_v*'))[-1]
!git -C $REPO_DIR log --oneline -1
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pyproj shapely

In [ ]:
# 2. Earth Engine (same project as the main notebook)
import ee
ee.Authenticate()
ee.Initialize(project='gaurav-singh-007')
print('Earth Engine ready')

In [ ]:
# 3. Config: identical to the main notebook's Tata Motors config
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
config = Config(
    gee_project='gaurav-singh-007', output_dir='outputs', archetype='industrial',
    realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True, reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()
print('riparian ring width (m):', config.riparian_ring_width_m, '| early period:', config.net_change_early_years)

In [ ]:
# 4. Locate the two zones from the Tata manifests
import sys, os
sys.path.insert(0, os.getcwd())
from run_project_from_manifest import find_manifest_by_project_name, load_manifest, resolve_tile_paths
REPO_ROOT = os.path.dirname(os.getcwd())
PATHS = {}
for proj in ('TataMotors_Pimpri', 'TataMotors_Pimpri_Aquatic'):
    mp = find_manifest_by_project_name(REPO_ROOT, proj)
    man = load_manifest(mp)
    PATHS.update(dict(zip(man['tile_labels'], resolve_tile_paths(man, mp))))
TERRESTRIAL_LABEL, AQUATIC_LABEL = 'EMU_Deccan_forest', 'Lake_Suman'
print(TERRESTRIAL_LABEL, '->', PATHS[TERRESTRIAL_LABEL]); print(AQUATIC_LABEL, '->', PATHS[AQUATIC_LABEL])

In [ ]:
# 5. SMOKE TEST A: one terrestrial zone (writes outputs/smoke/EMU_Deccan_forest_audit.{csv,json,md})
import logging, time
logging.basicConfig(level=logging.WARNING)
from darukaa_reference import assess as A
zoneA = A.load_zone(config, registry, PATHS[TERRESTRIAL_LABEL], TERRESTRIAL_LABEL, 'terrestrial')
print('ecoregion id:', zoneA.eco_id)
providerA = A.EEProvider(config, registry)
rowsA = A.run_smoke_test(config, registry, zoneA, out_dir='outputs/smoke', provider=providerA)

In [ ]:
# 6. SMOKE TEST B: one aquatic zone (Lake_Suman, 8.47 ha, the largest Tata water body)
zoneB = A.load_zone(config, registry, PATHS[AQUATIC_LABEL], AQUATIC_LABEL, 'aquatic')
print('ecoregion id:', zoneB.eco_id)
providerB = A.EEProvider(config, registry)
rowsB = A.run_smoke_test(config, registry, zoneB, out_dir='outputs/smoke', provider=providerB)

In [ ]:
# 7. EE-vs-numpy PARITY: pulls small real rasters and recomputes the numpy definitions on the same pixels.
# DISCREPANCY = a real finding; HARNESS_ERROR = the check itself failed (not evidence about the indicators).
from darukaa_reference import parity
results = parity.run_parity(config, registry, zoneA, zoneB, out_dir='outputs/smoke')

In [ ]:
# 8. Optional E4 forest-loss validation on an EXTERNAL forested polygon (never enters Tata scoring).
# Set VALIDATION_GEOJSON to a GeoJSON/KML path of a forested polygon with >= 5 ha of >=30 % canopy baseline; leave None to skip.
VALIDATION_GEOJSON = None
if VALIDATION_GEOJSON:
    zoneV = A.load_zone(config, registry, VALIDATION_GEOJSON, 'external_forest_validation', 'terrestrial',
                        validation_label='external forested polygon: methodological validation only, NOT a Tata zone')
    rowsV = A.run_smoke_test(config, registry, zoneV, out_dir='outputs/smoke', only=['forest_loss_rate'])

In [ ]:
# 9. Zip everything to send back
import shutil
shutil.make_archive('/content/v028_smoke_results', 'zip', 'outputs/smoke')
from google.colab import files
files.download('/content/v028_smoke_results.zip')